# Tide Connection

Implementa una lógica de **Extend to Next** sobre la red de calles.

Cada calle toma su vértice terminal más cercano a la costa, prolonga su último segmento en la misma dirección y busca el primer choque contra un segmento de costa. La intersección se resuelve con rayo-segmento y se filtran candidatos con índice espacial.

Matemática mínima:
- Rayo: `r(t) = p0 + t d`, con `t >= 0`
- Segmento candidato: `s(u) = q0 + u e`, con `0 <= u <= 1`
- Choque:
  - `t = cross(q0 - p0, e) / cross(d, e)`
  - `u = cross(q0 - p0, d) / cross(d, e)`

Si una calle ya fue procesada, el notebook la reutiliza y no vuelve a correr.


In [ ]:
import os
from collections import defaultdict, deque
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from shapely import make_valid
from shapely.geometry import LineString, Point
from shapely.ops import unary_union

MPLCONFIGDIR = Path('/tmp/matplotlib')
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(MPLCONFIGDIR))

import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

def find_project_root(start: Path | None = None) -> Path:
    """Return the repository root that contains `data/` and `notebooks/`."""
    start = start or Path.cwd()
    for candidate in [start, *start.parents]:
        if (candidate / 'data').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise FileNotFoundError('No se encontro la raiz del proyecto.')


ROOT = find_project_root()
SHORE_PATH = ROOT / 'data/spatial/vector/nyc_shoreline/nyc_shoreline_tide_primary.geojson'
STREETS_PATH = ROOT / 'data/spatial/vector/streets/processed/lion_metrics.gpkg'
BOUNDARY_PATH = ROOT / 'data/spatial/vector/nyc_borough_boundary/nybb.geojson'
OUTPUT_PATH = ROOT / 'data/spatial/vector/streets/processed/lion_metrics_shore.gpkg'

TARGET_CRS = 2263
SEED_BUFFER_FT = 1200.0
MAX_EXTENSION_FT = 10000.0
SEEDS_PER_STATION = 3
SHORE_METHOD = 'extend_to_next_v1'
REQUIRED_SHORE_COLUMNS = {
    'shore_id',
    'shore_processed',
    'shore_seed_edge',
    'shore_method',
    'shore_graph_steps',
}


In [ ]:
def load_inputs():
    """Load shoreline, streets and boundary layers."""
    for path in (SHORE_PATH, STREETS_PATH, BOUNDARY_PATH):
        if not path.exists():
            raise FileNotFoundError(f'No existe el archivo esperado: {path}')

    shore = gpd.read_file(SHORE_PATH)
    streets = gpd.read_file(STREETS_PATH)
    boundary = gpd.read_file(BOUNDARY_PATH)

    shore = shore.copy()
    shore['geometry'] = shore.geometry.apply(make_valid)
    shore = shore.explode(index_parts=False).reset_index(drop=True)
    shore['station_id'] = shore['station_id'].astype('string')

    streets = streets.copy()
    streets['edge_id'] = pd.to_numeric(streets['edge_id'], errors='coerce').astype('int64')
    streets['u'] = pd.to_numeric(streets['u'], errors='coerce').astype('int64')
    streets['v'] = pd.to_numeric(streets['v'], errors='coerce').astype('int64')
    streets['length'] = pd.to_numeric(streets.get('length'), errors='coerce') if 'length' in streets.columns else np.nan
    streets['length'] = streets['length'].fillna(streets.geometry.length)
    streets['length'] = streets['length'].where(streets['length'] > 0, streets.geometry.length)

    boundary = boundary.copy()
    boundary['geometry'] = boundary.geometry.apply(make_valid)

    print(f'Input counts -> shore: {len(shore)}, streets: {len(streets)}, boundary: {len(boundary)}', flush=True)
    return shore, streets, boundary


def project(frame: gpd.GeoDataFrame, crs: int = TARGET_CRS) -> gpd.GeoDataFrame:
    """Project a GeoDataFrame only when needed."""
    if frame.crs is not None and frame.crs.to_epsg() == crs:
        return frame.copy()
    return frame.to_crs(crs)


def prepare_streets(streets: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Keep the attributes needed for label propagation and plotting."""
    frame = streets.copy()
    if 'length' in frame.columns:
        frame['edge_length_ft'] = pd.to_numeric(frame['length'], errors='coerce')
    else:
        frame['edge_length_ft'] = np.nan
    frame['edge_length_ft'] = frame['edge_length_ft'].fillna(frame.geometry.length)
    frame['edge_length_ft'] = frame['edge_length_ft'].where(frame['edge_length_ft'] > 0, 1.0)
    return frame


def longest_line(geom):
    """Return a single LineString from LineString/MultiLineString input."""
    if geom is None or geom.is_empty:
        return None
    if geom.geom_type == 'LineString':
        return geom
    if geom.geom_type == 'MultiLineString':
        return max(geom.geoms, key=lambda part: part.length, default=None)
    if geom.geom_type == 'GeometryCollection':
        lines = [part for part in geom.geoms if part.geom_type in {'LineString', 'MultiLineString'}]
        if not lines:
            return None
        flattened = []
        for part in lines:
            if part.geom_type == 'LineString':
                flattened.append(part)
            else:
                flattened.extend(list(part.geoms))
        return max(flattened, key=lambda part: part.length, default=None)
    return None


def coastward_endpoint(geom, shore_geom):
    """Pick the terminal vertex closest to the shoreline and its outgoing direction."""
    line = longest_line(geom)
    if line is None:
        return None
    coords = np.asarray(line.coords, dtype=float)
    if len(coords) < 2:
        return None

    start = coords[0]
    start_next = coords[1]
    end = coords[-1]
    end_prev = coords[-2]

    start_dist = Point(start).distance(shore_geom)
    end_dist = Point(end).distance(shore_geom)

    if start_dist <= end_dist:
        anchor = start
        prev = start_next
        terminal = 'start'
        distance = start_dist
    else:
        anchor = end
        prev = end_prev
        terminal = 'end'
        distance = end_dist

    direction = anchor - prev
    norm = np.linalg.norm(direction)
    if not np.isfinite(norm) or norm == 0:
        return None

    return {
        'anchor': anchor,
        'direction': direction / norm,
        'terminal': terminal,
        'distance_to_shore': float(distance),
    }


def segment_intersection(p0, p1, q0, q1, tol=1e-12):
    """Return the first point where two segments intersect, if any."""
    r = p1 - p0
    s = q1 - q0
    denom = r[0] * s[1] - r[1] * s[0]
    if abs(denom) < tol:
        return None

    qp = q0 - p0
    t = (qp[0] * s[1] - qp[1] * s[0]) / denom
    u = (qp[0] * r[1] - qp[1] * r[0]) / denom
    if t < 0 or t > 1 or u < 0 or u > 1:
        return None
    return float(t), p0 + t * r


def iter_line_parts(geom):
    """Yield line parts from a possibly multipart geometry."""
    if geom is None or geom.is_empty:
        return
    if geom.geom_type == 'LineString':
        yield geom
    elif geom.geom_type == 'MultiLineString':
        yield from geom.geoms
    elif geom.geom_type == 'GeometryCollection':
        for part in geom.geoms:
            yield from iter_line_parts(part)


def first_hit_on_shore(ray_start, ray_end, shore_candidates):
    """Pick the closest shoreline segment hit by the extension ray."""
    ray = LineString([tuple(ray_start), tuple(ray_end)])
    candidate_idx = list(shore_candidates.sindex.intersection(ray.bounds))
    best = None
    for idx in candidate_idx:
        row = shore_candidates.iloc[idx]
        station_id = str(row.station_id)
        geom = row.geometry
        for part in iter_line_parts(geom):
            coords = np.asarray(part.coords, dtype=float)
            if len(coords) < 2:
                continue
            for q0, q1 in zip(coords[:-1], coords[1:]):
                hit = segment_intersection(np.asarray(ray_start, dtype=float), np.asarray(ray_end, dtype=float), np.asarray(q0, dtype=float), np.asarray(q1, dtype=float))
                if hit is None:
                    continue
                t, point = hit
                dist = float(np.linalg.norm(point - np.asarray(ray_start, dtype=float)))
                if best is None or dist < best['shore_touch_distance_ft']:
                    best = {
                        'station_id': station_id,
                        'shore_touch_distance_ft': dist,
                        'shore_touch_point': Point(float(point[0]), float(point[1])),
                        'ray': ray,
                        't': t,
                    }
    return best


def build_seed_edges(streets_proj, shore_proj):
    """Find coastal seed streets and label them with the shoreline station they touch first."""
    shore_union = unary_union(shore_proj.geometry)
    shore_candidates = shore_proj[['station_id', 'geometry']].copy().reset_index(drop=True)
    shore_candidates['station_id'] = shore_candidates['station_id'].astype('string')

    buffer_geom = shore_union.buffer(SEED_BUFFER_FT)
    street_candidates = list(streets_proj.sindex.intersection(buffer_geom.bounds))
    nearshore = streets_proj.iloc[street_candidates][['edge_id', 'u', 'v', 'edge_length_ft', 'geometry']].copy()
    nearshore = nearshore.loc[nearshore.geometry.intersects(buffer_geom)].copy()
    if nearshore.empty:
        raise ValueError('No se encontraron calles cercanas a la costa para usar como semillas.')

    rows = []
    for row in nearshore.itertuples(index=False):
        anchor_data = coastward_endpoint(row.geometry, shore_union)
        if anchor_data is None:
            continue
        ray_end = anchor_data['anchor'] + anchor_data['direction'] * MAX_EXTENSION_FT
        hit = first_hit_on_shore(anchor_data['anchor'], ray_end, shore_candidates)
        if hit is None:
            fallback = gpd.GeoDataFrame(
                {'edge_id': [row.edge_id]},
                geometry=gpd.GeoSeries([Point(anchor_data['anchor'][0], anchor_data['anchor'][1])], crs=streets_proj.crs),
                crs=streets_proj.crs,
            )
            nearest = gpd.sjoin_nearest(
                fallback,
                shore_candidates[['station_id', 'geometry']],
                how='left',
                distance_col='shore_touch_distance_ft',
            )
            station_id = nearest.iloc[0]['station_id']
            touch_distance = float(nearest.iloc[0]['shore_touch_distance_ft']) if pd.notna(nearest.iloc[0]['shore_touch_distance_ft']) else np.nan
            touch_point = fallback.geometry.iloc[0]
            ray = LineString([tuple(anchor_data['anchor']), tuple(ray_end)])
        else:
            station_id = hit['station_id']
            touch_distance = hit['shore_touch_distance_ft']
            touch_point = hit['shore_touch_point']
            ray = hit['ray']

        rows.append(
            {
                'edge_id': int(row.edge_id),
                'u': int(row.u),
                'v': int(row.v),
                'station_id': str(station_id),
                'shore_touch_distance_ft': float(touch_distance) if pd.notna(touch_distance) else np.nan,
                'shore_seed_edge': True,
                'shore_seed_terminal': anchor_data['terminal'],
                'shore_seed_distance_ft': float(anchor_data['distance_to_shore']),
                'shore_touch_x': float(touch_point.x) if touch_point is not None else np.nan,
                'shore_touch_y': float(touch_point.y) if touch_point is not None else np.nan,
                'shore_ray': ray,
            }
        )

    seeds = pd.DataFrame(rows)
    if seeds.empty:
        raise ValueError('No se pudieron construir semillas de costa.')

    seeds = (
        seeds.sort_values(['station_id', 'shore_touch_distance_ft', 'edge_id'])
        .groupby('station_id', group_keys=False)
        .head(SEEDS_PER_STATION)
        .reset_index(drop=True)
    )
    print(f'  - seed candidates: {len(rows)} | seed edges used: {len(seeds)}', flush=True)
    return seeds, shore_union, shore_candidates


def build_node_adjacency(streets_proj):
    """Build an undirected node adjacency list for the street graph."""
    adjacency = defaultdict(list)
    for row in streets_proj[['u', 'v']].itertuples(index=False):
        u = int(row.u)
        v = int(row.v)
        adjacency[u].append(v)
        adjacency[v].append(u)
    return adjacency


def propagate_shore_ids(streets_proj, shore_proj, seed_edges):
    """Propagate shoreline labels over the street graph with multi-source BFS."""
    adjacency = build_node_adjacency(streets_proj)
    best_node: dict[int, tuple[int, str]] = {}
    queue = deque()

    for seed in seed_edges[['station_id', 'u', 'v']].drop_duplicates().itertuples(index=False):
        station_id = str(seed.station_id)
        for node in (int(seed.u), int(seed.v)):
            candidate = (0, station_id)
            if node not in best_node or candidate < best_node[node]:
                best_node[node] = candidate
                queue.append((node, station_id))

    while queue:
        node, station_id = queue.popleft()
        hops, _ = best_node[node]
        for neighbor in adjacency.get(node, []):
            candidate = (hops + 1, station_id)
            if neighbor not in best_node or candidate < best_node[neighbor]:
                best_node[neighbor] = candidate
                queue.append((neighbor, station_id))

    edge_rows = []
    for row in streets_proj[['edge_id', 'u', 'v']].itertuples(index=False):
        candidates = [best_node[node] for node in (int(row.u), int(row.v)) if node in best_node]
        if candidates:
            hops, station_id = min(candidates)
        else:
            hops, station_id = (np.nan, pd.NA)
        edge_rows.append((row.edge_id, station_id, hops))

    assignments = pd.DataFrame(edge_rows, columns=['edge_id', 'shore_id', 'shore_graph_steps'])
    result = streets_proj.merge(assignments, on='edge_id', how='left')

    missing_mask = result['shore_id'].isna()
    if missing_mask.any():
        fallback = gpd.sjoin_nearest(
            result.loc[missing_mask, ['edge_id', 'geometry']].copy(),
            shore_proj[['station_id', 'geometry']],
            how='left',
            distance_col='nearest_shore_distance_ft',
        )
        fallback['station_id'] = fallback['station_id'].astype('string')
        result = result.merge(
            fallback[['edge_id', 'station_id']].rename(columns={'station_id': 'fallback_station_id'}),
            on='edge_id',
            how='left',
        )
        result['shore_id'] = result['shore_id'].fillna(result['fallback_station_id'])
        result = result.drop(columns=['fallback_station_id'])

    result['shore_id'] = result['shore_id'].astype('string')
    result['shore_processed'] = result['shore_id'].notna()
    result['shore_seed_edge'] = result['edge_id'].isin(seed_edges['edge_id'])
    result['shore_method'] = SHORE_METHOD
    result = result.drop(columns=['edge_length_ft'], errors='ignore')
    result['shore_processed'] = result['shore_processed'].astype(bool)
    result['shore_seed_edge'] = result['shore_seed_edge'].astype(bool)
    print(f'  - unlabeled edges after BFS: {int(missing_mask.sum())}', flush=True)
    return result


def is_complete_output(frame: gpd.GeoDataFrame) -> bool:
    """Return True when a saved output already has shoreline labels."""
    return (
        REQUIRED_SHORE_COLUMNS.issubset(frame.columns)
        and frame['shore_method'].astype('string').eq(SHORE_METHOD).all()
        and frame['shore_processed'].fillna(False).all()
    )


def build_assignment(shore: gpd.GeoDataFrame, streets: gpd.GeoDataFrame):
    """Build the shoreline assignment in projected CRS and return the saved CRS copy."""
    original_crs = streets.crs
    print('Building shoreline assignment...', flush=True)
    shore_proj = project(shore, TARGET_CRS)
    streets_proj = project(prepare_streets(streets), TARGET_CRS)
    print('  - selecting coastal seed streets', flush=True)
    seed_edges, shore_union, shore_candidates = build_seed_edges(streets_proj, shore_proj)
    print('  - propagating labels across the street graph', flush=True)
    result = propagate_shore_ids(streets_proj, shore_proj, seed_edges)
    result['shore_touch_distance_ft'] = np.nan
    result['shore_seed_distance_ft'] = np.nan
    result['shore_touch_x'] = np.nan
    result['shore_touch_y'] = np.nan
    result = result.merge(
        seed_edges[['edge_id', 'shore_touch_distance_ft', 'shore_seed_distance_ft', 'shore_touch_x', 'shore_touch_y']],
        on='edge_id',
        how='left',
        suffixes=('', '_seed'),
    )
    for column in ['shore_touch_distance_ft', 'shore_seed_distance_ft', 'shore_touch_x', 'shore_touch_y']:
        seed_col = f'{column}_seed'
        if seed_col in result.columns:
            result[column] = result[column].fillna(result[seed_col])
            result = result.drop(columns=[seed_col])
    print('  - finished propagation', flush=True)
    if original_crs is not None and result.crs is not None and result.crs != original_crs:
        result = result.to_crs(original_crs)
    return result, shore_proj, shore_union, seed_edges


def load_or_build_assignment():
    """Reuse a completed output file when possible."""
    shore, streets, boundary = load_inputs()
    shore_proj = project(shore, TARGET_CRS)
    boundary_proj = project(boundary, TARGET_CRS)

    if OUTPUT_PATH.exists():
        candidate = gpd.read_file(OUTPUT_PATH)
        if is_complete_output(candidate):
            print(f'Reutilizando salida procesada: {OUTPUT_PATH}', flush=True)
            return candidate, shore_proj, boundary_proj, None
        print('La salida existe, pero esta incompleta. Se reconstruye.', flush=True)

    result, shore_proj, _, seed_edges = build_assignment(shore, streets)
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    result.to_file(OUTPUT_PATH, driver='GPKG')
    print(f'Salida escrita en: {OUTPUT_PATH}', flush=True)
    return result, shore_proj, boundary_proj, seed_edges


In [ ]:
result, shore_proj, boundary_proj, seed_edges = load_or_build_assignment()

print('Shore stations:', sorted(shore_proj['station_id'].astype(str).unique().tolist()))
print('Street edges:', len(result))
print('Processed:', bool(result['shore_processed'].fillna(False).all()))

summary = (
    result.groupby('shore_id', dropna=False)
    .agg(
        edges=('edge_id', 'size'),
        total_length_ft=('length', 'sum'),
    )
    .sort_values('edges', ascending=False)
)
summary


Input counts -> shore: 168, streets: 196904, boundary: 5
La salida existe, pero esta incompleta. Se reconstruye.
Building shoreline assignment...
  - selecting coastal seed streets


In [ ]:
seed_rows = result.loc[result['shore_seed_edge']].copy()
seed_rows[['edge_id', 'shore_id', 'shore_graph_steps', 'shore_touch_distance_ft']].head(10)


In [ ]:
palette = {
    station_id: plt.get_cmap('tab10')(i % 10)
    for i, station_id in enumerate(sorted(result['shore_id'].dropna().astype(str).unique()))
}

result_plot = result.to_crs(TARGET_CRS) if result.crs is not None and result.crs.to_epsg() != TARGET_CRS else result
seed_edges_plot = result_plot.loc[result_plot['shore_seed_edge']].copy()

fig, ax = plt.subplots(figsize=(12, 12))

boundary_proj.boundary.plot(ax=ax, color='#4f4f4f', linewidth=1.0, zorder=1)

for station_id, group in result_plot.groupby('shore_id'):
    group.plot(
        ax=ax,
        color=palette.get(str(station_id), '#7f7f7f'),
        linewidth=0.18,
        alpha=0.28,
        zorder=2,
    )

for station_id, group in shore_proj.groupby('station_id'):
    group.plot(
        ax=ax,
        color=palette.get(str(station_id), '#000000'),
        linewidth=2.2,
        zorder=4,
    )

if len(seed_edges_plot):
    seed_edges_plot.plot(ax=ax, color='black', linewidth=0.6, zorder=5)

labels = shore_proj.dissolve(by='station_id').geometry.representative_point()
for station_id, point in labels.items():
    ax.text(point.x, point.y, str(station_id), fontsize=9, weight='bold', color='black')

xmin, ymin, xmax, ymax = boundary_proj.total_bounds
pad_x = (xmax - xmin) * 0.03
pad_y = (ymax - ymin) * 0.03
ax.set_xlim(xmin - pad_x, xmax + pad_x)
ax.set_ylim(ymin - pad_y, ymax + pad_y)
ax.set_aspect('equal')
ax.set_axis_off()
ax.set_title('NYC streets connected to shoreline stations with Extend to Next', fontsize=15)
plt.tight_layout()
plt.show()
